In [51]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.simulation import predict_match, assign_thirds
from src.features import update_team_history, update_h2h
import matplotlib as plt
import numpy as np

In [52]:
#Load everything from earlier notebooks
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
scaler = joblib.load('scaler.joblib')

# Scale up draw threshold so ~25% of group stage games end in draws (model default ~0.37 is too low)
draw_threshold = draw_threshold * 1.5

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

In [53]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

squad_values = {
    "Mexico": 226, "South Korea": 154, "South Africa": 50, "Czech Republic": 200,
    "Canada": 230, "Switzerland": 360, "Qatar": 23, "Bosnia and Herzegovina": 80,
    "Brazil": 1100, "Morocco": 350, "Haiti": 10, "Scotland": 200,
    "United States": 310, "Paraguay": 100, "Australia": 40, "Turkey": 320,
    "Germany": 850, "Curaçao": 5, "Ivory Coast": 300, "Ecuador": 230,
    "Netherlands": 600, "Japan": 290, "Sweden": 210, "Tunisia": 45,
    "Belgium": 450, "Egypt": 130, "Iran": 50, "New Zealand": 20,
    "Spain": 1000, "Cape Verde": 30, "Saudi Arabia": 30, "Uruguay": 480,
    "France": 1200, "Senegal": 250, "Iraq": 10, "Norway": 450,
    "Argentina": 800, "Algeria": 180, "Austria": 240, "Jordan": 15,
    "Portugal": 1050, "DR Congo": 110, "Uzbekistan": 35, "Colombia": 280,
    "England": 1500, "Croatia": 300, "Ghana": 200, "Panama": 20
}

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0,
                     "GD": 0,
                     'GF': 0,
                     "GA": 0})
        
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[match.home_team], country_elo[match.away_team])
    print(f"{match.home_team} vs {match.away_team} ({x.home_score}-{x.away_score}, result: {x.outcome}), Prob: {x.probability}, diff: {x.diff}")
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GD'] += x.home_score - x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GD'] += x.away_score - x.home_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GF'] += x.home_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GF'] += x.away_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GA'] += x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GA'] += x.home_score
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5

    new_away, new_home = update_elo(S, match.neutral, match.K_factor, match.home_score, match.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    
    update_team_history(match, history_dict)
    update_h2h(match, h2h_dict)

group_stage_result = group_stage_result.sort_values(['Group', 'Points', 'GD', 'GF'], ascending=[True, False, False, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results


Mexico vs South Africa (3-2, result: Home Win), Prob: 0.8499000072479248, diff: 0.8022000193595886
South Korea vs Czech Republic (3-2, result: Home Win), Prob: 0.8406000137329102, diff: 0.7299000024795532
Canada vs Bosnia and Herzegovina (2-1, result: Home Win), Prob: 0.9351000189781189, diff: 0.8996999859809875
United States vs Paraguay (1-0, result: Home Win), Prob: 0.7142000198364258, diff: 0.5871999859809875
Qatar vs Switzerland (4-3, result: Home Win), Prob: 0.670199990272522, diff: 0.5741999745368958
Brazil vs Morocco (1-0, result: Home Win), Prob: 0.7878000140190125, diff: 0.6679999828338623
Haiti vs Scotland (0-0, result: Draw), Prob: 0.09390000253915787, diff: 0.46149998903274536
Australia vs Turkey (0-0, result: Draw), Prob: 0.024399999529123306, diff: 0.48750001192092896
Germany vs Curaçao (0-0, result: Draw), Prob: 0.12120000272989273, diff: 0.47760000824928284
Ivory Coast vs Ecuador (1-0, result: Home Win), Prob: 0.7822999954223633, diff: 0.6462000012397766
Netherlands vs 

In [54]:
#Get teams that move on to round of 32

top2 = group_stage_result.groupby('Group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('Group').nth(2).copy() #All teams that placed third (only 8 of them move on)

best8_third = third.sort_values(
    ['Points', 'GD', 'GF'], 
    ascending=[False, False, False]
).head(8)

thirds_slot_map = {
    'E': 'ABCDF',
    'I': 'CDFGH', 
    'A': 'CEFHI',
    'L': 'EHIJK',
    'D': 'BEFIJ',
    'G': 'AEHIJ',
    'B': 'EFGIJ',
    'K': 'DEIJL',
}

winners = {g: teams.iloc[0]['Team'] for g, teams in group_stage_result.groupby('Group')}
runners = {g: teams.iloc[1]['Team'] for g, teams in group_stage_result.groupby('Group')}

r32_pairings = [
    (runners['A'], runners['B']),
    (winners['F'], runners['C']),
    (winners['C'], runners['F']),
    (runners['E'], runners['I']),
    (winners['H'], runners['J']),
    (winners['J'], runners['H']),
    (runners['K'], runners['L']),
    (runners['D'], runners['G']),
]

available_thirds = {row.Group: row.Team for row in best8_third.itertuples()}
assignments = {}
used = set()

for winner_group, eligible_groups in thirds_slot_map.items():
    for g in eligible_groups:
        if g in available_thirds and g not in used:
            assignments[winner_group] = available_thirds[g]
            used.add(g)
            break

third_pairings = [(winners[g], third_team) for g, third_team in assignments.items()]

r32_rows = []
r16_teams = []

for home, away in r32_pairings + third_pairings:
    x=predict_match(home, away, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[home], country_elo[away])

    if x.outcome == "Home Win": 
        S, winner = 1, home
    elif x.outcome == "Away Win": 
        S, winner = 0, away
    else: 
        S = .5
        winner = home if x.diff > 0 else away

    r16_teams.append(winner)
    r32_rows.append({
        'home_team': home,
        'away_team': away,
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{home} {x.home_score} - {x.away_score} {away} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    
    
    
    
    


Czech Republic 7 - 0 Qatar → Czech Republic
Tunisia 1 - 1 Brazil → Tunisia
Scotland 5 - 4 Netherlands → Scotland
Germany 2 - 1 France → Germany
Spain 10 - 0 Jordan → Spain
Argentina 1 - 1 Uruguay → Argentina
Portugal 8 - 0 Ghana → Portugal
Australia 4 - 3 Belgium → Australia
Ecuador 2 - 1 South Korea → Ecuador
Norway 6 - 5 Morocco → Norway
Mexico 6 - 0 Curaçao → Mexico
Panama 1 - 1 Saudi Arabia → Panama
United States 1 - 1 Sweden → United States
New Zealand 4 - 3 Senegal → New Zealand
Colombia 1 - 1 Croatia → Colombia
